# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) and contains multiple record sets and fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset URL referencing the Croissant JSON-LD schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
List available record sets, fields, their `@id`s, and brief details.

In [ ]:
# Display all available record sets with their @id and some meta info

record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"  @id: {rs['@id']} | name: {rs.get('name', '<no name>')} | description: {rs.get('description', '<no description>')}")

    # For each record set, print its fields and their @id
    for rs in record_sets:
        print(f"\nFields for record set '@id: {rs['@id']}' ({rs.get('name', '<no name>')}):")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            # field may be {'@id': ...} or dict
            if isinstance(f, dict):
                field_id = f.get('@id', str(f))
            else:
                field_id = str(f)
            print(f"    Field @id: {field_id}")

## 3. Data Extraction
Select record sets to extract and load into DataFrames. All entities are referenced by their `@id`.

In [ ]:
# Collect record set @ids
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets defined. Cannot proceed with data extraction.")
    dataframes = {}
else:
    record_set_ids = [rs['@id'] for rs in record_sets]

    # Load records for each record set into a DataFrame
    dataframes = {}
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded {len(df)} records from record set @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering numeric fields and grouping data by categorical fields. All field access is done via `@id`.

In [ ]:
# Only perform EDA if dataframes is non-empty
if not dataframes:
    print("No loaded dataframes to analyze.")
else:
    # Select first available record set for demonstration
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"Analyzing record set: {selected_record_set_id}")
    # Attempt to automatically select a numeric column by dtype or by typical names
    possible_numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not possible_numeric_cols:
        # Heuristics if no numeric dtypes
        numeric_names = ['log_likelihood', 'coefficient', 'p_value', 'std_error', 'value', 'age', 'income']
        for name in numeric_names:
            for col in df.columns:
                if name in col.lower():
                    possible_numeric_cols.append(col)
    if not possible_numeric_cols:
        print("No suitable numeric field found for EDA.")
    else:
        numeric_field_id = possible_numeric_cols[0]
        print(f"Numeric field selected for analysis: {numeric_field_id}")

        try:
            df = df.copy()
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = df[numeric_field_id].mean()  # Use mean as a dynamic threshold
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"\nFiltered records with {numeric_field_id} > {threshold:.3f}:")
            display(filtered_df.head())

            # Normalize
            norm_col = f"{numeric_field_id}_normalized"
            filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, norm_col]].head())

            # Attempt to group by a field: look for typical group fields
            group_candidates = [col for col in df.columns if 'ward' in col.lower() or 'gender' in col.lower() or 'county' in col.lower() or 'group' in col.lower()]
            if group_candidates:
                group_field_id = group_candidates[0]
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
                display(grouped_df)
            else:
                print("No suitable grouping categorical field found.")
        except Exception as e:
            print("Exploratory analysis failed:", str(e))

## 5. Visualization
Visualize data distributions or relationships using fields identified above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No dataframes loaded; skipping visualization.")
else:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if not numeric_fields:
        # fallback to possible heuristic fields
        heuristic_fields = ['log_likelihood', 'coefficient', 'p_value', 'std_error', 'value', 'age', 'income']
        for col in df.columns:
            for hf in heuristic_fields:
                if hf in col.lower():
                    numeric_fields.append(col)
    if not numeric_fields:
        print("No numeric field found for visualization.")
    else:
        field = numeric_fields[0]
        df[field] = pd.to_numeric(df[field], errors='coerce')
        plt.figure(figsize=(6,4))
        sns.histplot(df[field], kde=True)
        plt.title(f"Distribution of {field}")
        plt.xlabel(field)
        plt.ylabel("Frequency")
        plt.show()

    # If a categorical field is available, create boxplot
    group_candidates = [col for col in df.columns if 'ward' in col.lower() or 'gender' in col.lower() or 'county' in col.lower() or 'group' in col.lower()]
    if numeric_fields and group_candidates:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_candidates[0]], y=df[numeric_fields[0]])
        plt.title(f"{numeric_fields[0]} by {group_candidates[0]}")
        plt.xlabel(group_candidates[0])
        plt.ylabel(numeric_fields[0])
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and visualizing the FAIR^2 dataset using the `mlcroissant` library with explicit references to all dataset entities using their `@id` fields. Further analyses can proceed by extending the provided workflows to additional record sets and fields.